In [2]:
import subprocess
import os

def run_makeblastdb(input_file, dbtype, out, parse_seqids=False, hash_index=False):
    """
    Run a makeblastdb command with the given parameters.
    
    :param input_file: Path to the input FASTA file
    :param dbtype: Database type (nucl or prot)
    :param out: Path and name of the output database
    :param parse_seqids: Whether to parse sequence IDs (default: False)
    :param hash_index: Whether to create a hash index (default: False)
    :return: None
    """
    # Ensure the output directory exists
    os.makedirs(os.path.dirname(out), exist_ok=True)

    # Define the makeblastdb command
    makeblastdb_command = [
        "makeblastdb",
        "-in", input_file,
        "-dbtype", dbtype,
        "-out", out
    ]

    # Add optional parameters if specified
    if parse_seqids:
        makeblastdb_command.append("-parse_seqids")
    if hash_index:
        makeblastdb_command.append("-hash_index")

    try:
        # Run the makeblastdb command
        result = subprocess.run(makeblastdb_command, check=True, capture_output=True, text=True)
        
        # Print the output
        print(f"makeblastdb command for {input_file} executed successfully.")
        if result.stdout:
            print("Standard output:", result.stdout)
        
    except subprocess.CalledProcessError as e:
        print(f"An error occurred while running the makeblastdb command for {input_file}:")
        print("Error output:", e.stderr)
    except FileNotFoundError:
        print("Error: makeblastdb not found. Make sure it's in your system PATH or provide the full path to the executable.")



In [ ]:
# If the YAML configuration file is avaliable, use it to figure
# out what databases to create. Otherwise, go ahead and run
# makeblastdb commands manually as shown below.
import yaml
import os

yaml_file = os.getenv('NOVABROWSE_CONFIG', '')
if os.path.exists(yaml_file):
    with open(yaml_file, 'r') as f:
        config = yaml.safe_load(f)
    # Extract database information from the YAML configuration:
    # The ASSEMBLY_MAPPING section contains one subsection per species.
    # Each subsection contains 'assembly_acc' and a 'databases' list.
    # Each database entry has 'fasta_file', 'fasta_file_type', and 'type'
    # (either 'genome' or 'transcriptome').
    assembly_mapping = config.get('ASSEMBLY_MAPPING', {})
    for species, assembly_info in assembly_mapping.items():
        assembly_acc = assembly_info.get('assembly_acc')
        if not assembly_acc:
            print(f"Warning: Missing assembly_acc for {species}. Skipping.")
            continue

        databases = assembly_info.get('databases', [])

        for db in databases:
            fasta_file = db.get('fasta_file')
            fasta_file_type = db.get('fasta_file_type')
            db_type = db.get('type', 'transcriptome')
            if fasta_file and fasta_file_type:
                input_file = f"1_subject_sequences/{species}/{assembly_acc}/{fasta_file}"
                db_suffix = '_genome' if db_type == 'genome' else ''
                db_out_name = f"2_subject_blastdb/{species}_{assembly_acc}{db_suffix}"
                run_makeblastdb(input_file, fasta_file_type, db_out_name)
            else:
                print(f"Warning: Missing fasta_file or fasta_file_type for {species}. Skipping this database.")
    # Done, terminate.
    exit(0)

# Create BLAST databases

# S. cerevisiae - genome
run_makeblastdb(
    "1_subject_sequences\\s_cerevisiae\\GCF_000146045.2\\GCF_000146045.2_R64_genomic.fna",
    "nucl",
    "2_subject_blastdb\\s_cerevisiae_GCF_000146045.2_genome"
)

# S. cerevisiae - transcriptome
run_makeblastdb(
    "1_subject_sequences\\s_cerevisiae\\GCF_000146045.2\\rna.fna",
    "nucl",
    "2_subject_blastdb\\s_cerevisiae_GCF_000146045.2"
)

# S. pombe - transcriptome
run_makeblastdb(
    "1_subject_sequences\\s_pombe\\GCF_000002945.2\\rna.fna",
    "nucl",
    "2_subject_blastdb\\s_pombe_GCF_000002945.2"
)

# C. albicans - transcriptome
run_makeblastdb(
    "1_subject_sequences\\c_albicans\\GCF_000182965.3\\rna.fna",
    "nucl",
    "2_subject_blastdb\\c_albicans_GCF_000182965.3"
)